In [ ]:
import igraph
import squidpy as sq
import os
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc

In [ ]:
from xb.formatting import *
from xb.plotting import *
from xb.preprocessing import *
from xb.Spage_main import *
from xb.calculating import *

In [ ]:
os.chdir("/cluster3/yflu/STS/TMA/pipeline_output/separated/Banksy_py")

In [ ]:
import igraph
import spatialdata as sd

In [ ]:
zarr_path = "/cluster3/yflu/STS/TMA/full/zarr/TMA3.zarr"
sdata = sd.read_zarr(zarr_path)
sdata

In [ ]:
tx = sdata.points['transcripts']
print(tx.head())       # 看看有哪些列
print(tx.columns)

In [ ]:
nuclear_tx = tx[tx["overlaps_nucleus"] == 1]

In [ ]:
nuclear_tx

In [ ]:
adata = sdata.tables["table"]

In [ ]:
adata.write("/cluster3/yflu/STS/TMA/adata.h5ad")

In [ ]:
adata = sc.read_h5ad("/cluster3/yflu/STS/TMA/adata.h5ad")

In [ ]:
cell_merged_STS = pd.read_csv("/cluster3/yflu/STS/TMA/cell_merged_STS.csv",sep=',',header=0)
cell_merged_STS = cell_merged_STS[cell_merged_STS["TMA"].isin(["TMA3"])]
adata=adata[adata.obs.cell_id.isin(cell_merged_STS["Cell_ID"])].copy()
cell_mapping = pd.DataFrame({
    'order': adata.obs.cell_id.to_list(),
})
sort_mapping = cell_mapping.reset_index().set_index('order')
cell_merged_STS = cell_merged_STS[cell_merged_STS["Cell_ID"].isin(adata.obs.cell_id)]
cell_merged_STS['cell_order'] = cell_merged_STS['Cell_ID'].map(sort_mapping['index'])
cell_merged_STS = cell_merged_STS.sort_values('cell_order')
cell_merged_STS.index = adata.obs.index
adata.obs["Disease"] = pd.Series(cell_merged_STS["Disease"], dtype="category")
adata.obs["Sample"] = pd.Series(cell_merged_STS["Sample"], dtype="category")

In [ ]:
samples = adata.obs["Sample"].unique()

In [ ]:
cell_merged_STS

In [ ]:
from xb.domain_identification import *
import random
random_seed = 1234
np.random.seed(random_seed)

In [ ]:
samples

In [ ]:
samples[35:36]

In [ ]:
for sample in samples[34:36]:
    adata_pred = sc.read(f"/cluster3/yflu/STS/TMA/separated/{sample}_subset.h5ad")
    cell_merged_STS["cell_index"] = adata.obs.index
    cell_merged_STS_sub = cell_merged_STS[cell_merged_STS["Cell_ID_new"].isin(adata_pred.obs.index)].copy()
    cell_merged_STS_sub.index = cell_merged_STS_sub["Cell_ID_new"]
    adata_pred.obs["Cell_ID"] = pd.Series(cell_merged_STS_sub["Cell_ID"], dtype="category")
    adata_pred.obs["cell_order"] = pd.Series(cell_merged_STS_sub["cell_order"], dtype="category")
    adata_pred.obs["cell_index"] = pd.Series(cell_merged_STS_sub["cell_index"], dtype="category")
    adata_pred.obs.index = pd.Series(cell_merged_STS_sub["cell_index"], dtype="object")
    adata_1 = adata[adata.obs.cell_id.isin(adata_pred.obs["Cell_ID"])].copy()
    adata_1.obs["expressed_genes"] = (adata_1.X > 0).sum(axis=1)
    adata_1.obs["sample"] = "TMA3"
    adata_1 = adata_1[:, adata_1.var.index.isin(adata_pred.var.index)].copy()
    adata_pred.obs.index.name = None
    adata_1.X = adata_pred.X
    adata_1.obs["louvain_labels"] = pd.Series(adata_pred.obs["louvain_labels"], dtype="category")
    adata_1.obsm["umap"] = adata_pred.obsm["X_umap"]
    adata_1.obs["x_centroid"] = adata_1.obsm["spatial"][:, 0]  # X-coordinates
    adata_1.obs["y_centroid"] = adata_1.obsm["spatial"][:, 1]  # Y-coordinates
    adata_1.obs["unique_cell_id"] = adata_1.obs["cell_id"]
    '''Input File'''
    banksy_params={'resolutions':[.9], # clustering resolution for Leiden clustering
    'pca_dims':[20], # number of dimensions to keep after PCA
    'lambda_list':[.8],# lambda
    'k_geom':15, # 15 spatial neighbours
    'max_m':1, # use AGF
    'nbr_weight_decay':"scaled_gaussian", # can also be "reciprocal", "uniform" or "ranked"
    'cluster_algorithm':'leiden'}
    adata_1,adata_1_banksy=domains_by_banksy(adata_1,plot_path=f"/cluster3/yflu/STS/TMA/separated/{sample}/",banksy_params=banksy_params,save=False)
    adata_1.write(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_preprocessed.h5ad")

In [ ]:
sample = "T1091"
adata_pred = sc.read(f"/cluster3/yflu/STS/TMA/separated/{sample}_subset.h5ad")
cell_merged_STS["cell_index"] = adata.obs.index
cell_merged_STS_sub = cell_merged_STS[cell_merged_STS["Cell_ID_new"].isin(adata_pred.obs.index)].copy()
cell_merged_STS_sub.index = cell_merged_STS_sub["Cell_ID_new"]
adata_pred.obs["Cell_ID"] = pd.Series(cell_merged_STS_sub["Cell_ID"], dtype="category")
adata_pred.obs["cell_order"] = pd.Series(cell_merged_STS_sub["cell_order"], dtype="category")
adata_pred.obs["cell_index"] = pd.Series(cell_merged_STS_sub["cell_index"], dtype="category")
adata_pred.obs.index = pd.Series(cell_merged_STS_sub["cell_index"], dtype="object")
adata_1 = adata[adata.obs.cell_id.isin(adata_pred.obs["Cell_ID"])].copy()
adata_1.obs["expressed_genes"] = (adata_1.X > 0).sum(axis=1)
adata_1.obs["sample"] = "TMA3"
adata_1 = adata_1[:, adata_1.var.index.isin(adata_pred.var.index)].copy()
adata_pred.obs.index.name = None
adata_1.X = adata_pred.X
adata_1.obs["louvain_labels"] = pd.Series(adata_pred.obs["louvain_labels"], dtype="category")
adata_1.obsm["umap"] = adata_pred.obsm["X_umap"]
adata_1.obs["x_centroid"] = adata_1.obsm["spatial"][:, 0]  # X-coordinates
adata_1.obs["y_centroid"] = adata_1.obsm["spatial"][:, 1]  # Y-coordinates
adata_1.obs["unique_cell_id"] = adata_1.obs["cell_id"]
'''Input File'''
banksy_params={'resolutions':[.9], # clustering resolution for Leiden clustering
'pca_dims':[20], # number of dimensions to keep after PCA
'lambda_list':[.8],# lambda
'k_geom':15, # 15 spatial neighbours
'max_m':1, # use AGF
'nbr_weight_decay':"scaled_gaussian", # can also be "reciprocal", "uniform" or "ranked"
'cluster_algorithm':'leiden'}
adata_1,adata_1_banksy=domains_by_banksy(adata_1,plot_path=f"/cluster3/yflu/STS/TMA/separated/{sample}/",banksy_params=banksy_params,save=False)
adata_1.write(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_preprocessed.h5ad")

In [ ]:
pd.Series(cell_merged_STS_sub["cell_index"], dtype="object")

In [ ]:
adata_pred.obs.index

In [ ]:
sample = "T1091"
adata_1 = sc.read_h5ad(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_preprocessed.h5ad")

In [ ]:
adata_1

In [ ]:
import igraph
import scanpy as sc
import os
os.chdir("/cluster3/yflu/STS/TMA/pipeline_output/separated/Banksy_py")
from xb.domain_identification import *
sample = "T1416"
adata_1 = sc.read_h5ad(f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_preprocessed.h5ad")
banksy_params={'resolutions':[.9], # clustering resolution for Leiden clustering
    'pca_dims':[20], # number of dimensions to keep after PCA
    'lambda_list':[.8],# lambda
    'k_geom':15, # 15 spatial neighbours
    'max_m':1, # use AGF
    'nbr_weight_decay':"scaled_gaussian", # can also be "reciprocal", "uniform" or "ranked"
    'cluster_algorithm':'leiden'}
adata_1=domains_by_banksy(adata_1,banksy_params=banksy_params,save=False,plot_path=f"/cluster3/yflu/STS/TMA/separated/{sample}/")

In [ ]:
f"/cluster3/yflu/STS/TMA/separated/{sample}/"

In [ ]:
sample

In [ ]:
index = list(samples).index("CH3")
index

In [ ]:
samples[49:50]